In [1]:
import os,sys
import torch,re
import time
from tqdm import tqdm
from transformers import AutoModelForSpeechSeq2Seq, AutoProcessor
import librosa 

os.chdir("..")
print("localetion : ",os.getcwd())

localetion :  E:\PROJECT\GITHUB\GITHUB\audio_tosubtitles


In [2]:
# Kaynaklarda kullanılan temel ayarlar [4]
model_id = fr"C:\Users\yasin\.cache\huggingface\hub\models--openai--whisper-large-v3\snapshots\06f233fe06e710322aca913c1bc4249a0d71fce1"
device = "cuda:0" if torch.cuda.is_available() else "cpu"
torch_dtype = torch.float16 if torch.cuda.is_available() else torch.float32


In [3]:
# 1. Model ve Processor'ı Doğrudan Yükleme [1]
model = AutoModelForSpeechSeq2Seq.from_pretrained(
    model_id, torch_dtype=torch_dtype, low_cpu_mem_usage=True, use_safetensors=True
).to(device)

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/1259 [00:00<?, ?it/s]

In [6]:
processor = AutoProcessor.from_pretrained(model_id)

In [7]:

# Not: Ses dosyasını yüklemek için kaynaklarda belirtilmeyen 'librosa' gibi 
# harici bir kütüphane kullanmanız gerekebilir (Bu bilgi kaynak dışıdır).

audio_path = "except/data/dene.wav"
speech, sr = librosa.load(audio_path, sr=16000) # Whisper 16kHz ile çalışır

C:\Users\yasin\AppData\Local\Temp\ipykernel_24144\2911029046.py:5: UserWarning: PySoundFile failed. Trying audioread instead.
  speech, sr = librosa.load(audio_path, sr=16000) # Whisper 16kHz ile çalışır
C:\Users\yasin\AppData\Roaming\Python\Python313\site-packages\librosa\core\audio.py:184: FutureWarning: librosa.core.audio.__audioread_load
	Deprecated as of librosa version 0.10.0.
	It will be removed in librosa version 1.0.
  y, sr_native = __audioread_load(path, offset, duration, dtype)


In [8]:
generate_kwargs = {
    "max_length": 448,  # Uyarıyı kapatan kritik satır
    "num_beams": 1,
    "condition_on_prev_tokens": False, # Önceki tokenlara bağımlılığı kapatır [1]
    "compression_ratio_threshold": 1.35,
    "temperature": (0.0, 0.2, 0.4, 0.6, 0.8, 1.0), # Temperature fallback stratejisi [1], [3]
    "logprob_threshold": -1.0,
    "no_speech_threshold": 0.5,
    "return_timestamps": True, # Zaman damgalarını tahmin etmesini sağlar [2]
    "language":"english"
}

In [25]:
# 2. Parçalama Mantığı (Chunking) [2, 3]
# Whisper 30 saniyelik pencerelerle çalıştığı için veriyi buna göre böleceğiz.
chunk_length_s = 30
samples_per_chunk = chunk_length_s * sr
total_chunks = (len(speech) // samples_per_chunk) + 1
print(f"Toplam {total_chunks} parça işlenecek...")


def text_Totimestamps(text,start_time=0) :
        pattern = r"<\|(\d+\.\d+)\|>(.*?)<\|(\d+\.\d+)\|>"
        matches = re.findall(pattern, text)
        # İstenen format: [[[zaman damgası], [text]]]
        formatted_output = [{"timestamp":(float(m[0])+start_time, float(m[2])+start_time),
                             "text":(m[1].strip())} for m in matches]

        return formatted_output


Toplam 226 parça işlenecek...


In [10]:
# 3. Manuel İlerleme Çubuğu ile Döngü
with tqdm(total=total_chunks) as pbar:
    final_text=[]
    full_transcript=[]
    for s,i in enumerate(range(0, len(speech), samples_per_chunk)):
        # Ses parçasını al
        chunk = speech[i : i + samples_per_chunk]
        
        # Processor: Ses verisini modelin anlayacağı Mel-spektrograma çevirir [1, 5]
        input_features = processor(chunk, sampling_rate=sr, return_tensors="pt").input_features
        input_features = input_features.to(device).to(torch_dtype)
        
        # Model.generate: Metin üretim aşaması [1]
        with torch.no_grad():
            predicted_ids = model.generate(input_features,**generate_kwargs) # Dil seçimi [6]
            
        # Decode: Sayısal çıktıları metne dönüştürür
        transcription = processor.batch_decode(*predicted_ids, skip_special_tokens=True,decode_with_timestamps=True)
        formatted_output=text_Totimestamps(transcription[0],start_time=s*30)

        torch.cuda.empty_cache()
        # Her parça bittiğinde ilerleme çubuğunu güncelle

        final_text+=formatted_output
        pbar.update(1)

        
        

print("\nİşlem Tamamlandı.")

  0%|                                                                          | 0/226 [00:00<?, ?it/s]

ERROR! Session/line number was not unique in database. History logging moved to new session 8


  0%|▎                                                                 | 1/226 [00:02<10:34,  2.82s/it]

start_time :  0


  1%|▌                                                                 | 2/226 [00:04<08:24,  2.25s/it]

...

start_time :  6690


100%|███████████████████████████████████████████████████████████████▋| 225/226 [18:50<00:04,  4.93s/it]

start_time :  6720


100%|████████████████████████████████████████████████████████████████| 226/226 [18:51<00:00,  5.01s/it]

start_time :  6750

İşlem Tamamlandı.


In [14]:
deneme=final_text.copy()

# çoklamayı sil

## version 1

In [20]:
def remove_duplicates(data):
    end_element=""
    for s,element in tqdm(enumerate(data)):
        if end_element and element["text"].replace(" ","").lower() == end_element["text"].replace(" ","").lower():
            print("element : ",element)
            print("end_element : ",end_element)
            try:
                data.remove(element)
                data.remove(end_element)
            except:
                pass
            element["timestamp"]=(end_element["timestamp"][0],element["timestamp"][1])
            data.insert(s-1,element)
        else:
            end_element=element

    return data

In [23]:
import numpy as np 

In [24]:
np.save("try_data.npy",final_text,)

In [21]:
deneme=remove_duplicates(deneme)
deneme

1182it [00:00, 240231.98it/s]

element :  {'timestamp': (4017.0, 4018.0), 'text': 'Annual...'}
end_element :  {'timestamp': (4009.0, 4016.0), 'text': 'Annual...'}
element :  {'timestamp': (4095.52, 4098.66), 'text': 'Pretty woman, talk a while.'}
end_element :  {'timestamp': (4091.42, 4094.92), 'text': 'Pretty woman, talk a while.'}
element :  {'timestamp': (4126.0, 4127.8), 'text': "I'll stay."}
end_element :  {'timestamp': (4119.8, 4124.8), 'text': "I'll stay."}
element :  {'timestamp': (4179.8, 4180.8), 'text': 'Break!'}
end_element :  {'timestamp': (4178.8, 4179.8), 'text': 'Break!'}
element :  {'timestamp': (4483.0, 4484.0), 'text': 'Hello.'}
end_element :  {'timestamp': (4482.0, 4483.0), 'text': 'Hello.'}
element :  {'timestamp': (4486.0, 4487.0), 'text': 'Hello.'}
end_element :  {'timestamp': (4482.0, 4483.0), 'text': 'Hello.'}
element :  {'timestamp': (4487.0, 4488.0), 'text': 'Hello.'}
end_element :  {'timestamp': (4482.0, 4483.0), 'text': 'Hello.'}
element :  {'timestamp': (4504.0, 4506.0), 'text': 'Yes.'}

[{'timestamp': (0.0, 29.98), 'text': 'Transcription by ESO. Translation by —'},
 {'timestamp': (30.0, 59.980000000000004), 'text': 'Oh, my God!'},
 {'timestamp': (60.0, 62.0), 'text': 'New Jersey?'},
 {'timestamp': (62.0, 64.0), 'text': 'Austria.'},
 {'timestamp': (64.0, 66.0), 'text': 'Austria?'},
 {'timestamp': (66.0, 68.0), 'text': 'Wow, wow, wow.'},
 {'timestamp': (68.0, 70.0), 'text': 'Have a good day, my friend.'},
 {'timestamp': (70.0, 72.0), 'text': "Let's put on some more Barbie clothes."},
 {'timestamp': (72.0, 74.0), 'text': "I don't think so."},
 {'timestamp': (90.0, 119.98), 'text': "We'll be right back."},
 {'timestamp': (120.0, 149.98), 'text': 'Thank you.'},
 {'timestamp': (150.0, 179.98), 'text': "We'll be right back."},
 {'timestamp': (180.0, 180.32), 'text': 'Yes.'},
 {'timestamp': (183.16, 184.5), 'text': "That's right, but not less."},
 {'timestamp': (185.44, 186.48), 'text': 'Sausage Slick.'},
 {'timestamp': (186.6, 188.06), 'text': 'Come on, Sausage Slick.'},
 {'

## version 2

In [9]:
def text_Totimestamps(text,start_time=0) :
        print("start_time : ",start_time)
        pattern = r"<\|(\d+\.\d+)\|>(.*?)<\|(\d+\.\d+)\|>"
        matches = re.findall(pattern, text)
        # İstenen format: [[[zaman damgası], [text]]]
        formatted_output = [{"timestamp":(float(m[0])+start_time, float(m[2])+start_time),
                             "text":(m[1].strip())} for m in matches]

        return formatted_output

In [16]:
end_text='See you cry'

In [17]:
start_time=0


pattern = r"<\|(\d+\.\d+)\|>(.*?)<\|(\d+\.\d+)\|>"

matches = re.findall(pattern, example)

formatted_output=[]

for m in matches:
    if m[1].strip() == end_text:
        print("burda")
    formatted_output.append({"timestamp":(float(m[0])+start_time, float(m[2])+start_time),
                     "text":(m[1].strip())})

formatted_output

burda


[{'timestamp': (0.0, 5.0), 'text': "I'll cry"},
 {'timestamp': (9.0, 12.0), 'text': 'When I see you again'},
 {'timestamp': (12.0, 16.0), 'text': 'See you cry'},
 {'timestamp': (17.0, 20.0), 'text': "I'll hold you"}]

In [21]:
final_text

[{'timestamp': (0.0, 16.78),
  'text': "It's been a long day without you, my friend"},
 {'timestamp': (16.78, 22.74),
  'text': "And I'll tell you all about it when I see you again"},
 {'timestamp': (22.74, 28.82),
  'text': "We've come a long way from where we began"},
 {'timestamp': (28.82, 29.98), 'text': "Oh, I'll"},
 {'timestamp': (30.0, 34.74),
  'text': 'Tell you all about it when I see you again'},
 {'timestamp': (34.74, 37.7), 'text': 'When I see you again'},
 {'timestamp': (37.7, 41.480000000000004), 'text': 'Damn, who knew?'},
 {'timestamp': (41.74, 42.980000000000004), 'text': 'All the planes we flew'},
 {'timestamp': (42.980000000000004, 44.74),
  'text': "Good things we've been through"},
 {'timestamp': (44.74, 47.480000000000004),
  'text': "That I'd be standing right here talking to you"},
 {'timestamp': (47.480000000000004, 48.86), 'text': 'About another path'},
 {'timestamp': (48.86, 51.08),
  'text': 'I know we love to hit the road and laugh'},
 {'timestamp': (51.08,

# MODEL TEST

In [3]:
from model import WhisperLargeV3
from tools import data_to_srt

In [4]:
WhisperModel=WhisperLargeV3()

Loading weights:   0%|          | 0/1259 [00:00<?, ?it/s]

In [5]:
#WhisperModel.Load_model()

In [8]:
WhisperModel.chunk_length=15

In [6]:
#WhisperModel.language="turkish"

In [7]:
speech, sr = WhisperModel.Load_file("except/data/seeyou.mp3")

In [9]:
result=WhisperModel.preadiction(speech=speech, sr=sr)

  0%|                                                                           | 0/16 [00:00<?, ?it/s][transformers] The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
[transformers] A custom logits processor of type <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> has been passed to `.generate()`, but it was also created in `.generate()`, given its parameterization. The custom <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> will take precedence. Please check the docstring of <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> to see related `.generate()` flags.
[transformers] A custom logits processor of type <class 'transformers.generation.logits_process.SuppressTokensAtBeginLogitsProcessor'> has been passed to `.generate

In [8]:
data_to_srt(result,output_file="mahsun.srt")

In [9]:
# Bu kod kaynaklarda bulunmamaktadır.
from moviepy import VideoFileClip



In [10]:
video = VideoFileClip("mahsun.ts")
video.audio.write_audiofile("mahsun.mp3") # Sesi mp3 olarak kaydeder

MoviePy - Writing audio in mahsun.mp3


MoviePy - Done.


In [15]:
example={'timestamp': [0.0, 15.0], 'text': 'Damn, who knew?'}
for i in result:
     if i["text"] == example["text"]:
         example["timestamp"][1]=i["timestamp"][1]

In [17]:
example

{'timestamp': [0.0, 72.0], 'text': 'Damn, who knew?'}

[{'timestamp': (0.0, 20.0),
  'text': "It's been a long day without you my friend And I'll tell you all about it"},
 {'timestamp': (30.0, 42.24),
  'text': "it when i see you again we've come a long way from where we began oh i'll tell you all about it"},
 {'timestamp': (42.24, 50.0),
  'text': 'when i see you again when i see you again'},
 {'timestamp': (60.0, 61.46), 'text': 'Damn, who knew?'},
 {'timestamp': (61.74, 62.98), 'text': 'All the planes we flew'},
 {'timestamp': (62.98, 64.74), 'text': "Good things we've been through"},
 {'timestamp': (64.74, 66.5), 'text': "Then I'll be standing right here"},
 {'timestamp': (66.5, 67.46), 'text': 'Talking to you'},
 {'timestamp': (67.46, 68.86), 'text': 'About another path'},
 {'timestamp': (68.86, 71.1),
  'text': 'I know we love to hit the road and laugh'},
 {'timestamp': (71.1, 73.46000000000001),
  'text': "But something told me that it wouldn't last"},
 {'timestamp': (73.46000000000001, 74.7), 'text': 'Had to switch up'},
 {'timesta